In [1]:
import torch
device = torch.device("cpu")

In [2]:
from transformers import pipeline
classifier = pipeline("sentiment-analysis")
classifier(["I've been waiting for a Hugging Face course my whole life.","this is a great course for learning transformers"])


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9982948899269104},
 {'label': 'POSITIVE', 'score': 0.9996901750564575}]

## `Tokenizer -> Model -> Post Processing`

1. Transformers models cant process raw text directly. so the convert the text inputs in to numbers. that the model can make sense of it. so we use tokenizer.

- splitting the input into words, subwords or symbols that are called tokens
- Mapping each token to an integer.
- Adding additional inputs that may be usefull to the model

All this preprocessing neds to be done in exactly the same way when model was pretrained on.

to do this we use `Autotokenizer` call and its `from_pretrained()` method.

In [3]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
tokenizer

BertTokenizer(name_or_path='distilbert-base-uncased-finetuned-sst-2-english', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [4]:
raw_inputs = [
    "Some employers in US will ruin the employee's life",
    "I am a victim of it"
]

In [5]:
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
inputs


{'input_ids': tensor([[  101,  2070, 12433,  1999,  2149,  2097, 10083,  1996,  7904,  1005,
          1055,  2166,   102],
        [  101,  1045,  2572,  1037,  6778,  1997,  2009,   102,     0,     0,
             0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]])}

we can download out pretrained model the same way we did with our tokenizer. Transformers provices an AutoModel class which also has a `from_pretrained()` method

In [6]:
from transformers import AutoModel

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModel.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
pre_classifier.weight | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
outputs = model(**inputs)
outputs.last_hidden_state.shape

torch.Size([2, 13, 768])

The Model heads take the high-dimensional vector of hidden states as input and project them onto a different dimension.

`Model Input` -> `embiddings` -> `layers` -> `hidden states` -> `head` -> `model output`

The embeddings layer converts each input ID in the tokenized input into a vector that represents the associated token. The subsequent layers manipulate those vectors using attention mechanism to produce the final representation of the sentences.

There are naby different architectures availabe in transformenrs with each one desigined around tackling a specfic task,
- AutoModel
- AutoFoecausalLM
- AutoForMaskedLM
- AutoForMultipleChoice
- AutoForQuestionAnswering
- AutoForSequenceClassification
- AutoForTokenClassification and many

In [8]:
from transformers import AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
outputs = model(**inputs)
outputs.logits.shape

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

torch.Size([2, 2])

## Postprocessing the output

In [9]:
print(outputs.logits)

tensor([[ 4.6281, -3.7913],
        [ 4.4144, -3.5901]], grad_fn=<AddmmBackward0>)


this are not probabilities but logits that are outcomes of lastlayer. this need to be converted to probabilites using `softmax` 

In [10]:
import torch

predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(predictions)

tensor([[9.9978e-01, 2.2049e-04],
        [9.9967e-01, 3.3384e-04]], grad_fn=<SoftmaxBackward0>)


In [11]:
model.config.id2label

{0: 'NEGATIVE', 1: 'POSITIVE'}

## Models:


## Tokenizers

They are the core componentes of NLP tasks. and there are many types of tokenizers :

- Word-based
- Character-based
- Subword Tokenization
- wordPiece, as used in BERT
- Byte-level BPE as used in GPT-2
- SentencePiece or Unigram, as used in several multilingual models


In [12]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-cased")

In [13]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [14]:
x = "This Hugging face course is very helpfull"
tokenizer(x)

{'input_ids': [101, 1188, 20164, 10932, 1339, 1736, 1110, 1304, 14739, 1233, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [15]:
t= tokenizer.tokenize(x)

In [16]:
ids = tokenizer.convert_tokens_to_ids(t)
ids

[1188, 20164, 10932, 1339, 1736, 1110, 1304, 14739, 1233]

### Decoding

In [17]:
decoded_string = tokenizer.decode(ids)
decoded_string

'This Hugging face course is very helpfull'

### Handeling Multiple Sequences


In [18]:
from huggingface_hub import login
login()

In [19]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model= AutoModelForSequenceClassification.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

C:\Users\ShashidharReddyMaram\Desktop\Hugging_face_repo\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ShashidharReddyMaram\.cache\huggingface\hub\models--Qwen--Qwen3-0.6B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-0.6B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
sequence = "I've been waiting for a HuggingFace course my whole life."
tokens = tokenizer.tokenize(sequence)
tokens

['I',
 "'ve",
 'Ġbeen',
 'Ġwaiting',
 'Ġfor',
 'Ġa',
 'ĠH',
 'ugging',
 'Face',
 'Ġcourse',
 'Ġmy',
 'Ġwhole',
 'Ġlife',
 '.']

In [27]:
ids = tokenizer.convert_tokens_to_ids(tokens)
ids

input_data=torch.tensor(ids)
model(input_data)

RuntimeError: The size of tensor a (16) must match the size of tensor b (128) at non-singleton dimension 3

In [29]:
tokenized_inputs = tokenizer(sequence, return_tensors="pt")
print(tokenized_inputs["input_ids"])

tensor([[   40,  3003,  1012,  8580,   369,   264,   472, 35268, 16281,  3308,
           847,  4361,  2272,    13]])


In [33]:

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)

input_ids = torch.tensor([ids])
print(input_ids)
output = model(input_ids)
print("logits",output.logits)

tensor([[   40,  3003,  1012,  8580,   369,   264,   472, 35268, 16281,  3308,
           847,  4361,  2272,    13]])
logits tensor([[-0.2812,  1.5469]], dtype=torch.bfloat16, grad_fn=<IndexBackward0>)


Batching allows model to work with multiple sequneces in a single turn. but the problem, is that sometimes the sentences are not same leghts, but the tesors needs to be in rectangle shape. so we use padding the make the both sentes of same size .

In [37]:
batched_ids = [ids,ids]
# bathched_ids
for x in batched_ids:
    print(len(x))

14
14


In [44]:
tokenizer.pad_token_id

151643

In [42]:
sequence1_ids = [[200, 200, 200]]
sequence2_ids = [[200, 200]]
batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id],
]

In [45]:
batched_ids

[[200, 200, 200], [200, 200, 151643]]

In [48]:
print(model(torch.tensor(sequence1_ids)).logits)
print(model(torch.tensor(sequence2_ids)).logits)
# print(model(torch.tensor(batched_ids)).logits)

tensor([[ 2.2812, -1.3594]], dtype=torch.bfloat16, grad_fn=<IndexBackward0>)
tensor([[ 2.2812, -1.3594]], dtype=torch.bfloat16, grad_fn=<IndexBackward0>)


## Putting all together

Transformers functions handells all this together in a very high level function

In [53]:
from transformers import AutoTokenizer
checkpoint =  "distilbert-base-uncased-finetuned-sst-2-english"
tokeinzer =  AutoTokenizer.from_pretrained(checkpoint)
sequence = "I've been waiting for a HuggingFace course my whole life."
model_inputs = tokenizer(sequence)
model_inputs

{'input_ids': [40, 3003, 1012, 8580, 369, 264, 472, 35268, 16281, 3308, 847, 4361, 2272, 13], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [55]:
sequences = ["I've been waiting for a HuggingFace course my whole life.", "So have I!"]

model_inputs = tokenizer(sequences)
model_inputs

{'input_ids': [[40, 3003, 1012, 8580, 369, 264, 472, 35268, 16281, 3308, 847, 4361, 2272, 13], [4416, 614, 358, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1]]}

In [56]:
# Will pad the sequences up to the maximum sequence length
model_inputs = tokenizer(sequences, padding="longest")

# Will pad the sequences up to the model max length
# (512 for BERT or DistilBERT)
model_inputs = tokenizer(sequences, padding="max_length")

# Will pad the sequences up to the specified max length
model_inputs = tokenizer(sequences, padding="max_length", max_length=8)

In [57]:
# Will pad the sequences up to the maximum sequence length
model_inputs = tokenizer(sequences, padding="longest")

# Will pad the sequences up to the model max length
# (512 for BERT or DistilBERT)
model_inputs = tokenizer(sequences, padding="max_length")

# Will pad the sequences up to the specified max length
model_inputs = tokenizer(sequences, padding="max_length", max_length=8)

In [63]:
sequence = "learning and following huggingface cources can change my life and get new oppurtuniex in my carier"

model_inputs = tokenizer(sequence)
model_inputs
model_inputs["input_ids"]

[20981,
 323,
 2701,
 305,
 35268,
 1564,
 2081,
 1603,
 646,
 2297,
 847,
 2272,
 323,
 633,
 501,
 3995,
 5639,
 15705,
 327,
 304,
 847,
 1803,
 1268]

In [65]:
tokens = tokenizer.tokenize(sequence)
tokens

['learning',
 'Ġand',
 'Ġfollowing',
 'Ġh',
 'ugging',
 'face',
 'Ġcour',
 'ces',
 'Ġcan',
 'Ġchange',
 'Ġmy',
 'Ġlife',
 'Ġand',
 'Ġget',
 'Ġnew',
 'Ġopp',
 'urt',
 'uni',
 'ex',
 'Ġin',
 'Ġmy',
 'Ġcar',
 'ier']

In [67]:
ids  = tokenizer.convert_tokens_to_ids(tokens)
ids

[20981,
 323,
 2701,
 305,
 35268,
 1564,
 2081,
 1603,
 646,
 2297,
 847,
 2272,
 323,
 633,
 501,
 3995,
 5639,
 15705,
 327,
 304,
 847,
 1803,
 1268]

In [68]:
tokenizer.decode(ids)

'learning and following huggingface cources can change my life and get new oppurtuniex in my carier'

In [69]:
print(tokenizer.decode(model_inputs["input_ids"]))

learning and following huggingface cources can change my life and get new oppurtuniex in my carier


### Final Code block of alll the above code.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
sequences = ["I've been waiting for a HuggingFace course my whole life.", "So have I!"]

tokens = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")
output = model(**tokens)

## Optimized Inference Deployment

TGI, vLLM and llama.cpp server similar purposes that make them better suited for differnet use cases.

Mmory management and perforamnce

### TGI

- desigined to be stable and predictable in production using fixed sequence lengths to keep memory usage consistent
- TGI manage memory using Flash Attention 2 and continuous batching techniques.
- This means it can process attention calculation very efficently and keep the GPU busy by constantly feeding it work.
- The system can move parts of the model between CPU and GPU when needed, which helps handle larger models.

- Flash attention is a technique that optmizes the attention mechanim in transformers models by addressing memory bandwidth bottlenecks.

### Flash Attention

This is a technique that optimizes attention mechanism in transofmer models by adding memory bandwidth bottlenecks
 - the key innovation is how it manages memory transfers beteween High Bandwidth Memory(HBM) and faster SRAM cache. 
 - traditional attention repeaterly transfers data between HBM and SRAM creating bottlenecks leaving GPU idle, while flash attention loads data once in to SRAM and performs all calculations there minimizing expensive menort transfers.


 While the benefits are most significant during traning, flash attention's reduce VRAM usage and improved efficency make it valuable for infernece as well.

### vLLM

- Uses PagedAttention just like how how computer manages memory pages
- the model stores attention keys and value(K,V caching) for each generated token to reduce redundant computations.
- the KV cache can become enormous especially with long sequences or multiple concurrent requests.
- vLLM's Key innovation :
    - Memory Paging: instead of treating the KV cache as a one large block its divide into fixed-size "pages"
    - "non-coutiguous Storage:" Pages dont need to be contiguously in GPU memory, allowing for more flexible moemory allocation
    - Page Table Management: A page table tracjs which pages belong to which sequence, enabling effiencent lookup and access
    - Memory Sharing: For operations like parallel sampling pages storing the KV cache for the prompt can be shared across multiple sequences
    


### llama.cpp:
- C++ implementation, desigined to run on cpu machines but with optional GPU acceleration and is ideal for resources constrained enviroments.
- it used a quantization techniques to reduce model size and memory requirements for good performance.


### Quantization: 

- Reduces the precision of model weights from 32-bit or 16-bit floating point to lower precison formats like 8-bit integers. 4 bit or even lower

Key Quantization: 
-  Multiple Quantization levels:
- GGML/GGUF Format: Uses custom tensor formats optimized for quantizes inference
- Mixed precision: Can apply different quantizatinon levels to differnt parts of the models
- Hardware-Specfic Optimizations: Includes optmized code pats for various CPU architectures (AVX2,AVX-512,NEON)

This approach enables running billion-parametr models on consumer hardware with limited memory, making it perfect for locak deployments and edge devices



## Deployment and integration:

TGI:  excels in enterprice-level deployment with its production-ready features
vLLM: takes a more flexible, developer-frendly approach to deployment, 
llama.cpp: simple and portable 
